# 🔁 04 — Cohort retention & engagement

Les utilisateurs reviennent-ils ? Combien de temps ?

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.data_loader import load_sessions, load_transactions
from src.preprocessing import enrich
from src.datamarts import (
    build_cohort_retention_pivot, build_cohort_long, build_new_vs_returning
)
from src.utils import set_style

set_style()
sessions = enrich(load_sessions(), load_transactions())

## Cohort retention pivot

In [ ]:
cohort = build_cohort_retention_pivot(sessions)
cohort.iloc[:8, :8]  # aperçu 8x8

## Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))
sns.heatmap(cohort.iloc[:12, :12], annot=True, fmt='.0f', cmap='YlGnBu',
            cbar_kws={'label': '% rétention'}, vmin=0, vmax=100, linewidths=0.4)
ax.set_title("Cohort retention")
plt.show()

## Format long pour Looker

In [ ]:
long = build_cohort_long(sessions)
long.head(15)

## New vs Returning

In [ ]:
build_new_vs_returning(sessions)

💡 **Insight** : les utilisateurs récurrents convertissent ~3x mieux que les nouveaux. Investir dans la rétention (email, fidélité) est plus rentable que l'acquisition pure.

## Engagement par niveau

In [ ]:
engagement = sessions.groupby('engagement_level', observed=True).agg(
    sessions=('session_id', 'count'),
    conversion_rate_pct=('converted', lambda s: 100*s.mean()),
    avg_revenue=('session_revenue_usd', 'mean'),
).round(2)
engagement